<a href="https://colab.research.google.com/github/mithunkumarsr/NeurIPS-MAS-2026/blob/main/Lab_1_Hierarchical_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1: Hierarchical Memory Architecture
**NeurIPS 2026 Education Track: Multi-Agent Orchestration**

**Author:** Mithun Kumar S R (Google)

In this lab, we operationalize the Hierarchical Memory Tuple:
$$ M = \langle M_{work}, M_{ep}, M_{sem} \rangle $$

We will build $M_{ep}$ (Episodic Memory) using ChromaDB to grant the agent infinite recall of historical state-action pairs without degrading its $M_{work}$ (Working Memory) token window.


In [ ]:
# Install required libraries
!pip install -q chromadb numpy

import chromadb
import uuid
import time
import numpy as np


### 1. Initializing Episodic Memory ($M_{ep}$)
 We initialize a continuous vector space to store the agent's discrete execution logs.


In [ ]:
# Initialize the local database client
client = chromadb.Client()

# Create a collection for our agent's autobiographical memory
episodic_memory = client.get_or_create_collection(
    name="agent_alpha_episodes"
)
print("Episodic Memory initialized successfully.")

Episodic Memory initialized successfully.



### 2. The Geometry of $M_{ep}$: Storing an Episode
When an agent executes an action, we map the discrete text $X$ into a continuous high-dimensional vector space $E: X \rightarrow \mathbb{R}^d$. We also tag it with a timestamp to calculate temporal decay later.


In [ ]:
def store_episode(task: str, action: str, outcome: str):
    """Embeds and stores a state-action-outcome tuple into M_ep."""
    doc_id = str(uuid.uuid4())
    current_time = time.time()

    # The document is what ChromaDB embeds and searches against
    document_text = f"Task: {task} | Action: {action} | Result: {outcome}"

    episodic_memory.add(
        documents=[document_text],
        metadatas=[{
            "task_type": "database_query",
            "success": outcome == "OK",
            "timestamp": current_time
        }],
        ids=[doc_id]
    )
    print(f"Episode {doc_id[:8]}... stored in M_ep.")

# Simulating a noisy enterprise environment
store_episode("Connect to Postgres", "psycopg2.connect(uri)", "504 Gateway Timeout")
store_episode("Connect to Postgres", "psycopg2.connect(uri, sslmode='require')", "OK")

Episode 183c874d... stored in M_ep.
Episode 2869de73... stored in M_ep.


### 3. Mathematical Forgetting: Temporal Decay
 To survive the curse of dimensionality, an agent must forget deprecated environments. We apply an exponential decay factor governed by $\lambda$ (the forgetting rate):
 $$ Score(O_{current}, S_{past}) = \cos(\theta) \times e^{-\lambda \Delta t} $$

In practice, we optimize this via metadata cutoff filtering.



In [ ]:
def retrieve_relevant_episodes(current_observation: str, k: int = 1, decay_days: int = 7):
    """Retrieves top-k similar past episodes, filtering out decayed memories."""
    # Apply hard temporal cutoff simulating lambda decay
    decay_window = decay_days * 24 * 60 * 60
    cutoff_time = time.time() - decay_window

    results = episodic_memory.query(
        query_texts=[current_observation],
        n_results=k,
        where={"timestamp": {"$gt": cutoff_time}}
    )

    if not results['documents'][0]:
        return "No relevant history found."

    return results['documents'][0][0]


### 4. Executing Agentic RAG
 Before the agent acts on $O_t$, it retrieves historical context.


In [ ]:
current_obs = "Postgres connection is failing with a timeout error."
injected_context = retrieve_relevant_episodes(current_obs)

print(f"Current Observation (O_t): {current_obs}")
print(f"Retrieved Context (R): {injected_context}")
print("\nConclusion: The agent can now inject this solution directly into M_work, achieving O(1) context scaling.")

Current Observation (O_t): Postgres connection is failing with a timeout error.
Retrieved Context (R): Task: Connect to Postgres | Action: psycopg2.connect(uri) | Result: 504 Gateway Timeout

Conclusion: The agent can now inject this solution directly into M_work, achieving O(1) context scaling.
